<a href="https://colab.research.google.com/github/faithmutanu49/-plp-da-a1-gikomba-stall-analysis/blob/main/wanjiku_stall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gikomba Stall Performance Analysis

**Client:** Wanjiku, fresh-produce stall, Gikomba Market
**Analyst:** [Faith Mutanu Mbithi] | **Cohort:** [1 million dev]

Wanjiku asked: *How is my business doing? Is it growing? What should I restock, and where should I focus?*
This notebook answers those questions using **core Python only** (no pandas or NumPy).

## Dataset

Two weeks of daily sales (KES) and a stock record for five products. The data cell below is exactly as provided.

In [8]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Summary statistics and best/worst days

**Approach:** Use `sum()`, `len()`, `max()` and `min()` for the totals. `list.index()` gives the position of the best and worst days. Positions 0-6 are week 1 and 7-13 are week 2, so the week number is `position // 7 + 1`. The day name comes from the same position in `days`.

In [9]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

best_idx = daily_sales.index(highest_sales)
worst_idx = daily_sales.index(lowest_sales)

best_day, best_week = days[best_idx], best_idx // 7 + 1
worst_day, worst_week = days[worst_idx], worst_idx // 7 + 1

print(f"Total sales   : KES {total_sales:,}")
print(f"Average daily : KES {average_sales:,.2f}")
print(f"Highest day   : KES {highest_sales:,} ({best_day}, week {best_week})")
print(f"Lowest day    : KES {lowest_sales:,} ({worst_day}, week {worst_week})")

Total sales   : KES 46,450
Average daily : KES 3,317.86
Highest day   : KES 5,600 (Sat, week 2)
Lowest day    : KES 1,950 (Tue, week 1)


## Task 2: Week totals and week-on-week growth

**Approach:** `week_totals()` slices the list into `[:7]` and `[7:14]`, sums each slice and returns both totals as a tuple. The result is unpacked into two variables, and growth is `(week 2 - week 1) / week 1 * 100`.

In [10]:
def week_totals(sales):
    """Return (week 1 total, week 2 total) for a list of 14 daily sales figures.

    Week 1 is the first 7 days (indices 0-6) and week 2 is the next 7 (indices 7-13).
    """
    return sum(sales[:7]), sum(sales[7:14])


week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print(f"Week 1 total : KES {week1_total:,}")
print(f"Week 2 total : KES {week2_total:,}")
print(f"Growth       : {growth_pct:+.1f}% week-on-week (KES {week2_total - week1_total:,} more)")

Week 1 total : KES 22,050
Week 2 total : KES 24,400
Growth       : +10.7% week-on-week (KES 2,350 more)


## Task 3: Classifying each day

**Approach:** `classify_day()` compares a day's sales with the average. More than 20% above (`> average * 1.2`) is "Strong", more than 20% below (`< average * 0.8`) is "Weak", and everything else is "Normal". I apply it to every day with a loop and count the results with the dictionary counting pattern (`counts[label] = counts.get(label, 0) + 1`).

In [11]:
def classify_day(amount, average):
    """Classify a day's sales relative to the average.

    Returns "Strong" if amount is more than 20% above average,
    "Weak" if more than 20% below average, otherwise "Normal".
    """
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    return "Normal"


day_labels = []
category_counts = {}
for day, amount in zip(days, daily_sales):
    label = classify_day(amount, average_sales)
    day_labels.append(label)
    category_counts[label] = category_counts.get(label, 0) + 1

print(f"Strong above KES {average_sales * 1.2:,.2f}; Weak below KES {average_sales * 0.8:,.2f}\n")
for i, (day, amount, label) in enumerate(zip(days, daily_sales, day_labels), start=1):
    print(f"Day {i:>2} ({day}): KES {amount:>6,}  ->  {label}")

print()
for label in ("Strong", "Normal", "Weak"):
    print(f"{label:<7}: {category_counts.get(label, 0)} days")

Strong above KES 3,981.43; Weak below KES 2,654.29

Day  1 (Mon): KES  2,300  ->  Weak
Day  2 (Tue): KES  1,950  ->  Weak
Day  3 (Wed): KES  2,800  ->  Normal
Day  4 (Thu): KES  2,100  ->  Weak
Day  5 (Fri): KES  3,600  ->  Normal
Day  6 (Sat): KES  5,200  ->  Strong
Day  7 (Sun): KES  4,100  ->  Strong
Day  8 (Mon): KES  2,500  ->  Weak
Day  9 (Tue): KES  2,200  ->  Weak
Day 10 (Wed): KES  3,100  ->  Normal
Day 11 (Thu): KES  2,400  ->  Weak
Day 12 (Fri): KES  3,900  ->  Normal
Day 13 (Sat): KES  5,600  ->  Strong
Day 14 (Sun): KES  4,700  ->  Strong

Strong : 4 days
Normal : 4 days
Weak   : 6 days


## Task 4: Weekdays versus weekends

**Approach:** Loop over `days` and `daily_sales` together. Sat and Sun go into a weekend list and the rest into a weekday list. I average each list and express the difference as a percentage of the weekday average.

In [12]:
weekday_sales = []
weekend_sales = []
for day, amount in zip(days, daily_sales):
    if day in ("Sat", "Sun"):
        weekend_sales.append(amount)
    else:
        weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len(weekday_sales)
weekend_avg = sum(weekend_sales) / len(weekend_sales)
weekend_premium_pct = (weekend_avg - weekday_avg) / weekday_avg * 100

print(f"Weekday average (Mon-Fri): KES {weekday_avg:,.2f}")
print(f"Weekend average (Sat-Sun): KES {weekend_avg:,.2f}")
print(f"Weekends sell {weekend_premium_pct:.1f}% more per day than weekdays")

Weekday average (Mon-Fri): KES 2,685.00
Weekend average (Sat-Sun): KES 4,900.00
Weekends sell 82.5% more per day than weekdays


## Task 5: Restock list and gross margins

**Approach:** `products_to_reorder()` loops through `stock.items()` and returns the names where `qty <= reorder`. `gross_margin_pct()` applies `(price - cost) / price * 100` to one product. I run the margin function over every product and store the results in a dictionary.

In [13]:
def products_to_reorder(stock_data):
    """Return a list of product names whose quantity is at or below the reorder level."""
    low_stock = []
    for product, info in stock_data.items():
        if info["qty"] <= info["reorder"]:
            low_stock.append(product)
    return low_stock


def gross_margin_pct(info):
    """Return gross margin as a percentage: (price - cost) / price * 100."""
    return (info["price"] - info["cost"]) / info["price"] * 100


reorder_list = products_to_reorder(stock)
margins = {product: gross_margin_pct(info) for product, info in stock.items()}

print("Products to reorder:")
for product in reorder_list:
    info = stock[product]
    print(f"  {product:<9} {info['qty']} kg in stock, reorder level {info['reorder']} kg "
          f"(short by {info['reorder'] - info['qty']} kg)")

print("\nGross margin by product:")
for product, margin in margins.items():
    print(f"  {product:<9} {margin:5.1f}%")

Products to reorder:
  Tomatoes  18 kg in stock, reorder level 25 kg (short by 7 kg)
  Sukuma    9 kg in stock, reorder level 15 kg (short by 6 kg)

Gross margin by product:
  Tomatoes   33.3%
  Onions     30.0%
  Sukuma     40.0%
  Potatoes   33.3%
  Avocado    42.9%


## Task 6: Stall Performance Report

**Approach:** All results from Tasks 1-5 are brought together with f-strings. Money uses the `:,` format specifier for thousands separators, and column widths (`:<`, `:>`) keep the report aligned.

In [14]:
width = 56
line = "=" * width

print(line)
print("STALL PERFORMANCE REPORT".center(width))
print("Wanjiku's Fresh Produce, Gikomba Market".center(width))
print(line)

print("\nSALES SUMMARY (14 days)")
print("-" * width)
print(f"{'Total sales':<28}{'KES ' + format(total_sales, ','):>28}")
print(f"{'Average daily sales':<28}{'KES ' + format(average_sales, ',.2f'):>28}")
print(f"{'Best day':<28}{f'KES {highest_sales:,} ({best_day}, Wk {best_week})':>28}")
print(f"{'Worst day':<28}{f'KES {lowest_sales:,} ({worst_day}, Wk {worst_week})':>28}")

print("\nGROWTH")
print("-" * width)
print(f"{'Week 1 total':<28}{'KES ' + format(week1_total, ','):>28}")
print(f"{'Week 2 total':<28}{'KES ' + format(week2_total, ','):>28}")
print(f"{'Week-on-week growth':<28}{format(growth_pct, '+.1f') + '%':>28}")

print("\nDAY CLASSIFICATION")
print("-" * width)
for label in ("Strong", "Normal", "Weak"):
    print(f"{label + ' days':<28}{category_counts.get(label, 0):>28}")

print("\nWEEKDAY vs WEEKEND")
print("-" * width)
print(f"{'Weekday average':<28}{'KES ' + format(weekday_avg, ',.2f'):>28}")
print(f"{'Weekend average':<28}{'KES ' + format(weekend_avg, ',.2f'):>28}")
print(f"{'Weekend premium':<28}{format(weekend_premium_pct, '.1f') + '%':>28}")

print("\nSTOCK & MARGINS")
print("-" * width)
print(f"{'Product':<12}{'Qty (kg)':>9}{'Reorder':>9}{'Margin':>9}{'Status':>14}")
for product, info in stock.items():
    status = "REORDER" if product in reorder_list else "OK"
    print(f"{product:<12}{info['qty']:>9}{info['reorder']:>9}{margins[product]:>8.1f}%{status:>14}")
print(line)

                STALL PERFORMANCE REPORT                
        Wanjiku's Fresh Produce, Gikomba Market         

SALES SUMMARY (14 days)
--------------------------------------------------------
Total sales                                   KES 46,450
Average daily sales                         KES 3,317.86
Best day                           KES 5,600 (Sat, Wk 2)
Worst day                          KES 1,950 (Tue, Wk 1)

GROWTH
--------------------------------------------------------
Week 1 total                                  KES 22,050
Week 2 total                                  KES 24,400
Week-on-week growth                               +10.7%

DAY CLASSIFICATION
--------------------------------------------------------
Strong days                                            4
Normal days                                            4
Weak days                                              6

WEEKDAY vs WEEKEND
--------------------------------------------------------
Weekday average

## Recommendations for Wanjiku

**Task 7 approach:** I turned the findings from Tasks 1-6 into five plain-language sentences, each backed by a number from the analysis.

Wanjiku, your business is growing: sales rose 10.7% from KES 22,050 in week 1 to KES 24,400 in week 2. Weekends are your engine, averaging KES 4,900 a day against KES 2,685 on weekdays (82.5% more), and all 4 Strong days fell on a Saturday or Sunday, so have full shelves by Friday evening. Reorder Sukuma (9 kg against a 15 kg level) and Tomatoes (18 kg against 25 kg) now, because both are below their reorder levels just before your best days. Push Avocado, your best earner at a 42.9% margin (KES 60 profit per kg), and hold off buying more Potatoes (55 kg against a 30 kg level) and Onions (40 kg against 20 kg), whose margins are only 33.3% and 30.0% and which already tie up cash. Finally, all 6 Weak days were weekdays below KES 2,654, so a small weekday offer on your fastest-moving items is the best way to lift them.